In [30]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [31]:
import torch
import torch.nn as nn
import math

# ----------------------------
# Positional Encoding
# ----------------------------
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=500):
        super().__init__()

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()

        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        self.pe = pe.unsqueeze(0)  # (1, max_len, d_model)

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :].to(x.device)


# ----------------------------
# Multi-Head Attention
# ----------------------------
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()

        assert d_model % num_heads == 0

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        self.Wq = nn.Linear(d_model, d_model)
        self.Wk = nn.Linear(d_model, d_model)
        self.Wv = nn.Linear(d_model, d_model)

        self.fc_out = nn.Linear(d_model, d_model)

    def forward(self, x, mask=None):
        B, T, C = x.shape

        Q = self.Wq(x)
        K = self.Wk(x)
        V = self.Wv(x)

        # Split heads
        Q = Q.view(B, T, self.num_heads, self.d_k).transpose(1, 2)
        K = K.view(B, T, self.num_heads, self.d_k).transpose(1, 2)
        V = V.view(B, T, self.num_heads, self.d_k).transpose(1, 2)

        # Attention scores
        scores = (Q @ K.transpose(-2, -1)) / math.sqrt(self.d_k)

        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))

        attn = torch.softmax(scores, dim=-1)

        out = attn @ V  # (B, heads, T, d_k)

        # Concatenate heads
        out = out.transpose(1, 2).contiguous().view(B, T, C)

        return self.fc_out(out)


# ----------------------------
# Feed Forward Network
# ----------------------------
class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Linear(d_ff, d_model)
        )

    def forward(self, x):
        return self.net(x)


# ----------------------------
# Encoder Layer
# ----------------------------
class EncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff):
        super().__init__()

        self.attn = MultiHeadAttention(d_model, num_heads)
        self.ffn = FeedForward(d_model, d_ff)

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x, mask=None):
        # Attention + Residual
        x = self.norm1(x + self.attn(x, mask))

        # FFN + Residual
        x = self.norm2(x + self.ffn(x))

        return x


# ----------------------------
# Transformer Encoder
# ----------------------------
class TransformerEncoder(nn.Module):
    def __init__(
        self,
        input_dim,
        d_model=128,
        num_heads=4,
        d_ff=256,
        num_layers=2,
        max_len=500
    ):
        super().__init__()

        # If Word2Vec dim != d_model
        self.input_proj = nn.Linear(input_dim, d_model) if input_dim != d_model else None

        self.pos_enc = PositionalEncoding(d_model, max_len)

        self.layers = nn.ModuleList([
            EncoderLayer(d_model, num_heads, d_ff)
            for _ in range(num_layers)
        ])

    def forward(self, x, mask=None):
        # Project if needed
        if self.input_proj:
            x = self.input_proj(x)

        x = self.pos_enc(x)

        for layer in self.layers:
            x = layer(x, mask)

        return x


# ----------------------------
# TEST (Run this in Kaggle)
# ----------------------------

    

In [32]:
batch_size = 2
seq_len = 10
embedding_dim = 100   # simulate Word2Vec
d_model = 128

model = TransformerEncoder(
        input_dim=embedding_dim,
        d_model=d_model,
        num_heads=4,
        d_ff=256,
        num_layers=2
    )

dummy_input = torch.randn(batch_size, seq_len, embedding_dim)

    # Optional mask (1 = keep, 0 = pad)
mask = torch.ones(batch_size, 1, 1, seq_len)

output = model(dummy_input, mask)

print("Input shape:", dummy_input.shape)
print("Output shape:", output.shape)

Input shape: torch.Size([2, 10, 100])
Output shape: torch.Size([2, 10, 128])
